In [1]:
import numpy.typing as npt
from dataclasses import dataclass
from typing import Optional
from dsp import physical_model, FS_MIN

@dataclass
class ParameterRange:
    """Define valid range for each synthesis parameter."""
    min: float
    max: float

    def clip(self, value: npt.NDArray) -> npt.NDArray:
        """Clip values to valid range."""
        return np.clip(value, self.min, self.max)

    def center(self) -> float:
        """Get center value of range."""
        return (self.min + self.max) / 2.0

# Parameter ranges
PARAM_RANGES = {
    'f0': ParameterRange(20.0, FS_MIN / 2.0),
    'pluck_position': ParameterRange(0.0, 1.0),
    'burst_gain': ParameterRange(0.0, 1.0),
    'dynamic_level': ParameterRange(0.0, 1.0),
    'a1': ParameterRange(-1.0, 0.0),
    'decay': ParameterRange(0.0, 1.0),
}

@dataclass
class LFOConfig:
    """Configuration for LFO modulation."""
    rate: float  # Hz
    amplitude: float  # 0 to 1 (as fraction of full parameter range)

    def generate(
        self,
        num_samples: int,
        center: float,
        param_range: ParameterRange,
        fs: int = FS_MIN
    ) -> npt.NDArray:
        """Generate LFO-modulated parameter values."""
        t = np.arange(num_samples) / fs
        lfo = np.sin(2 * np.pi * self.rate * t)
        range_span = param_range.max - param_range.min
        modulation_depth = self.amplitude * range_span / 2.0
        modulated = center + lfo * modulation_depth
        return param_range.clip(modulated)

@dataclass
class ExperimentConfig:
    """Complete experiment configuration."""
    # Required parameters
    duration: float
    trigger_rate: float

    # Optional parameters with defaults
    fs: int = FS_MIN
    f0_center: float = 440.0
    pluck_position_center: float = 0.5
    burst_gain_center: float = 0.5
    dynamic_level_center: float = 0.5
    a1_center: float = -0.5
    decay_center: float = 0.99
    f0_lfo: Optional[LFOConfig] = None
    pluck_position_lfo: Optional[LFOConfig] = None
    burst_gain_lfo: Optional[LFOConfig] = None
    dynamic_level_lfo: Optional[LFOConfig] = None
    a1_lfo: Optional[LFOConfig] = None
    decay_lfo: Optional[LFOConfig] = None
    lagrange_order: int = 5

    def get_num_samples(self) -> int:
        return int(self.duration * self.fs)

    def generate_triggers(self) -> npt.NDArray:
        num_triggers = int(self.duration * self.trigger_rate)
        trigger_interval = self.fs / self.trigger_rate
        return np.arange(num_triggers) * trigger_interval

    def generate_parameter(
        self,
        param_name: str,
        center: float,
        lfo: Optional[LFOConfig]
    ) -> npt.NDArray:
        num_samples = self.get_num_samples()
        param_range = PARAM_RANGES[param_name]

        if lfo is None:
            return np.full(num_samples, param_range.clip(np.array([center]))[0])
        else:
            return lfo.generate(num_samples, center, param_range, self.fs)

    def generate_all_parameters(self) -> dict:
        return {
            'num_samples': self.get_num_samples(),
            'trigger_samples': self.generate_triggers(),
            'f0': self.generate_parameter('f0', self.f0_center, self.f0_lfo),
            'pluck_position': self.generate_parameter('pluck_position', self.pluck_position_center, self.pluck_position_lfo),
            'burst_gain': self.generate_parameter('burst_gain', self.burst_gain_center, self.burst_gain_lfo),
            'dynamic_level': self.generate_parameter('dynamic_level', self.dynamic_level_center, self.dynamic_level_lfo),
            'a1': self.generate_parameter('a1', self.a1_center, self.a1_lfo),
            'decay': self.generate_parameter('decay', self.decay_center, self.decay_lfo),
            'fs': self.fs,
            'lagrange_order': self.lagrange_order,
        }

    def synthesize(self) -> npt.NDArray:
        params = self.generate_all_parameters()
        return physical_model(**params)

In [2]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from scipy import signal
import librosa
import numpy as np
import IPython.display as ipd

# Create output widget for plots and audio
output = widgets.Output()

# LFO controls
lfo_rate = widgets.FloatSlider(
    value=1.0,
    min=0.01,
    max=10.0,
    step=0.01,
    description='LFO Rate (Hz):',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)

lfo_amplitude = widgets.FloatSlider(
    value=0.3,
    min=0.0,
    max=1.0,
    step=0.01,
    description='LFO Amplitude:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)

# Parameter center sliders
f0_center = widgets.FloatSlider(
    value=220.0,
    min=20.0,
    max=FS_MIN / 2,
    step=1.0,
    description='F0 Center (Hz):',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)

pluck_position_center = widgets.FloatSlider(
    value=0.5,
    min=0.0,
    max=1.0,
    step=0.01,
    description='Pluck Position:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)

burst_gain_center = widgets.FloatSlider(
    value=0.5,
    min=0.0,
    max=1.0,
    step=0.01,
    description='Burst Gain:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)

dynamic_level_center = widgets.FloatSlider(
    value=0.5,
    min=0.0,
    max=1.0,
    step=0.01,
    description='Dynamic Level:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)

a1_center = widgets.FloatSlider(
    value=-0.5,
    min=-1.0,
    max=0.0,
    step=0.01,
    description='a1 (Filter):',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)

decay_center = widgets.FloatSlider(
    value=0.995,
    min=0.0,
    max=1.0,
    step=0.001,
    description='Decay:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)

trigger_rate = widgets.FloatSlider(
    value=4.0,
    min=0.1,
    max=20.0,
    step=0.1,
    description='Trigger Rate (Hz):',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)

# Modulation checkboxes
f0_modulated = widgets.Checkbox(
    value=False,
    description='Modulate F0',
    style={'description_width': '150px'}
)

pluck_position_modulated = widgets.Checkbox(
    value=False,
    description='Modulate Pluck Position',
    style={'description_width': '150px'}
)

burst_gain_modulated = widgets.Checkbox(
    value=False,
    description='Modulate Burst Gain',
    style={'description_width': '150px'}
)

dynamic_level_modulated = widgets.Checkbox(
    value=False,
    description='Modulate Dynamic Level',
    style={'description_width': '150px'}
)

a1_modulated = widgets.Checkbox(
    value=False,
    description='Modulate a1',
    style={'description_width': '150px'}
)

decay_modulated = widgets.Checkbox(
    value=False,
    description='Modulate Decay',
    style={'description_width': '150px'}
)

# Generate button
generate_button = widgets.Button(
    description='Generate Audio',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)

def generate_audio(b):
    """Callback for generate button."""
    with output:
        clear_output(wait=True)

        print("Generating audio...")

        # Create LFO config
        lfo = LFOConfig(rate=lfo_rate.value, amplitude=lfo_amplitude.value)

        # Create experiment config
        config = ExperimentConfig(
            duration=8.0,  # Fixed 8 seconds
            trigger_rate=trigger_rate.value,
            f0_center=f0_center.value,
            pluck_position_center=pluck_position_center.value,
            burst_gain_center=burst_gain_center.value,
            dynamic_level_center=dynamic_level_center.value,
            a1_center=a1_center.value,
            decay_center=decay_center.value,
            # Apply LFO only if checkbox is ticked
            f0_lfo=lfo if f0_modulated.value else None,
            pluck_position_lfo=lfo if pluck_position_modulated.value else None,
            burst_gain_lfo=lfo if burst_gain_modulated.value else None,
            dynamic_level_lfo=lfo if dynamic_level_modulated.value else None,
            a1_lfo=lfo if a1_modulated.value else None,
            decay_lfo=lfo if decay_modulated.value else None,
        )

        # Synthesize
        audio = config.synthesize()
        params = config.generate_all_parameters()

        print("Estimating F0 with librosa...")
        # Extract F0 using librosa's pyin algorithm
        f0_librosa, voiced_flag, voiced_probs = librosa.pyin(
            audio.astype(np.float32),
            fmin=librosa.note_to_hz('C2'),  # ~65 Hz
            fmax=librosa.note_to_hz('C7'),  # ~2093 Hz
            sr=config.fs,
            frame_length=2048,
            hop_length=512
        )

        # Create time axis for librosa F0 (downsampled due to hop_length)
        hop_length = 512
        time_librosa = librosa.frames_to_time(
            np.arange(len(f0_librosa)),
            sr=config.fs,
            hop_length=hop_length
        )

        # Plot
        fig = plt.figure(figsize=(16, 12))
        gs = fig.add_gridspec(5, 2, hspace=0.35, wspace=0.3)

        time = np.arange(params['num_samples']) / config.fs

        # F0 Comparison plot (top, full width)
        ax_f0 = fig.add_subplot(gs[0, :])
        ax_f0.plot(time, params['f0'], linewidth=2, color='tab:blue',
                   label='Ground Truth F0', alpha=0.8)
        ax_f0.plot(time_librosa, f0_librosa, linewidth=1.5, color='tab:red',
                   label='Librosa F0 (pyin)', alpha=0.7, linestyle='--')
        ax_f0.set_ylabel('F0 (Hz)', fontsize=11)
        ax_f0.set_xlabel('Time (s)', fontsize=10)
        ax_f0.set_title('F0 Tracking: Ground Truth vs Librosa Estimation', fontsize=12, fontweight='bold')
        ax_f0.legend(loc='upper right', fontsize=10)
        ax_f0.grid(True, alpha=0.3)
        ax_f0.set_ylim([0, min(1000, np.nanmax(f0_librosa) * 1.2) if not np.all(np.isnan(f0_librosa)) else 1000])

        # Parameter plots (left column)
        param_info = [
            ('pluck_position', 'Pluck Position', pluck_position_modulated.value),
            ('dynamic_level', 'Dynamic Level', dynamic_level_modulated.value),
            ('a1', 'a1 (Filter)', a1_modulated.value),
            ('decay', 'Decay', decay_modulated.value),
        ]

        for i, (name, label, is_modulated) in enumerate(param_info):
            ax = fig.add_subplot(gs[i+1, 0])
            color = 'tab:blue' if is_modulated else 'tab:gray'
            ax.plot(time, params[name], linewidth=1.5, color=color, alpha=0.8)
            ax.set_ylabel(label, fontsize=10)
            ax.set_xlabel('Time (s)', fontsize=9)
            ax.grid(True, alpha=0.3)
            title = f'{label} {"(MODULATED)" if is_modulated else "(static)"}'
            ax.set_title(title, fontsize=10)

            # Add range lines
            param_range = PARAM_RANGES[name]
            ax.axhline(param_range.min, color='red', linestyle='--', alpha=0.3, linewidth=0.8)
            ax.axhline(param_range.max, color='red', linestyle='--', alpha=0.3, linewidth=0.8)

        # Audio analysis (right column)
        # Waveform
        ax1 = fig.add_subplot(gs[1, 1])
        ax1.plot(time, audio, linewidth=0.5, color='tab:green')
        ax1.set_ylabel('Amplitude', fontsize=10)
        ax1.set_xlabel('Time (s)', fontsize=9)
        ax1.set_title('Waveform', fontsize=11)
        ax1.grid(True, alpha=0.3)

        # Spectrogram
        ax2 = fig.add_subplot(gs[2:4, 1])
        f, t, Sxx = signal.spectrogram(audio, config.fs, nperseg=1024, noverlap=512)
        im = ax2.pcolormesh(t, f, 10 * np.log10(Sxx + 1e-10),
                            shading='gouraud', cmap='viridis', vmin=-80, vmax=0)
        ax2.set_ylabel('Frequency (Hz)', fontsize=10)
        ax2.set_xlabel('Time (s)', fontsize=9)
        ax2.set_title('Spectrogram', fontsize=11)
        ax2.set_ylim([0, min(4000, config.fs/2)])
        plt.colorbar(im, ax=ax2, label='Power (dB)')

        # Spectrum
        ax3 = fig.add_subplot(gs[4, 1])
        freqs = np.fft.rfftfreq(len(audio), 1/config.fs)
        spectrum_db = 20 * np.log10(np.abs(np.fft.rfft(audio)) + 1e-10)
        ax3.plot(freqs, spectrum_db, linewidth=0.5, color='tab:orange')
        ax3.set_xlabel('Frequency (Hz)', fontsize=9)
        ax3.set_ylabel('Magnitude (dB)', fontsize=10)
        ax3.set_title('Frequency Spectrum', fontsize=11)
        ax3.set_xlim([0, min(4000, config.fs/2)])
        ax3.grid(True, alpha=0.3)

        fig.suptitle(f'Karplus-Strong Synthesis (8s, {trigger_rate.value:.1f} triggers/s)',
                     fontsize=14, fontweight='bold')
        plt.show()

        # Compute F0 tracking metrics
        # Interpolate librosa F0 to match ground truth time grid
        f0_librosa_interp = np.interp(time, time_librosa,
                                       np.nan_to_num(f0_librosa, nan=0.0))

        # Only compute error where librosa detected pitch
        valid_mask = f0_librosa_interp > 0
        if np.any(valid_mask):
            abs_error = np.abs(params['f0'][valid_mask] - f0_librosa_interp[valid_mask])
            relative_error = abs_error / params['f0'][valid_mask] * 100
            mean_abs_error = np.mean(abs_error)
            mean_rel_error = np.mean(relative_error)
        else:
            mean_abs_error = np.nan
            mean_rel_error = np.nan

        # Audio stats
        print(f"\n{'='*60}")
        print(f"Audio Statistics:")
        print(f"  Duration: {len(audio)/config.fs:.2f} s")
        print(f"  Sample rate: {config.fs} Hz")
        print(f"  Peak amplitude: {np.max(np.abs(audio)):.3f}")
        print(f"  RMS: {np.sqrt(np.mean(audio**2)):.3f}")
        print(f"  Number of triggers: {len(params['trigger_samples'])}")
        print(f"\nF0 Tracking Accuracy:")
        print(f"  Mean absolute error: {mean_abs_error:.2f} Hz")
        print(f"  Mean relative error: {mean_rel_error:.2f}%")
        print(f"  Voiced frames detected: {np.sum(~np.isnan(f0_librosa))}/{len(f0_librosa)} ({100*np.sum(~np.isnan(f0_librosa))/len(f0_librosa):.1f}%)")
        print(f"{'='*60}\n")

        # Play audio
        display(ipd.Audio(audio, rate=config.fs))

# Connect button
generate_button.on_click(generate_audio)

# Layout
lfo_box = widgets.VBox([
    widgets.HTML("<h3>LFO Settings</h3>"),
    lfo_rate,
    lfo_amplitude,
])

general_box = widgets.VBox([
    widgets.HTML("<h3>General Settings</h3>"),
    trigger_rate,
])

params_box = widgets.VBox([
    widgets.HTML("<h3>Parameter Centers</h3>"),
    widgets.HBox([f0_center, f0_modulated]),
    widgets.HBox([pluck_position_center, pluck_position_modulated]),
    widgets.HBox([burst_gain_center, burst_gain_modulated]),
    widgets.HBox([dynamic_level_center, dynamic_level_modulated]),
    widgets.HBox([a1_center, a1_modulated]),
    widgets.HBox([decay_center, decay_modulated]),
])

control_panel = widgets.VBox([
    lfo_box,
    general_box,
    params_box,
    widgets.HTML("<br>"),
    generate_button,
])

# Display
display(control_panel, output)

print("Interactive Karplus-Strong Synthesizer with F0 Tracking")
print("Adjust parameters and click 'Generate Audio' to synthesize")

Output()

Interactive Karplus-Strong Synthesizer with F0 Tracking
Adjust parameters and click 'Generate Audio' to synthesize
